# Introduction to catsim

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/douglasrizzo/catsim/blob/main/notebooks/01_introduction_to_catsim.ipynb)

`catsim` is a toolkit for computerized adaptive testing (CAT). The modern
package architecture has four main pieces:

- `ItemBank`: calibrated item parameters and item-level utilities
- `CatEngine`: stepwise CAT execution for manual or third-party use
- `SimulationRunner`: repeated CAT sessions for research studies
- `SimulationResult`: aggregate outputs from a simulation run

This notebook is the shortest path through those ideas. It shows one small
simulation, one inspected session, and one compact manual CAT loop.

## Installation

Install the package with:

```bash
pip install -U catsim
```

If you are using Colab, restart the runtime after installation if imports
fail in the first execution.

## Setup

We will keep the examples small, deterministic, and fast enough to run in a
notebook session.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

from catsim import plot
from catsim.engine import CatEngine, RunContext
from catsim.estimation import NumericalSearchEstimator
from catsim.initialization import FixedPointInitializer, RandomInitializer
from catsim.item_bank import ItemBank
from catsim.selection import MaxInfoSelector
from catsim.simulation import SimulationRunner
from catsim.stopping import MinErrorStopper, TestLengthStopper

## Generate an item bank

A CAT needs calibrated items. In `catsim`, an item bank stores the item
parameters and a few precomputed values that only depend on the items
themselves.

In [ ]:
item_bank = ItemBank.generate_item_bank(120, itemtype="4PL", seed=7)
print(f"Bank size: {item_bank.n_items}")
print("First three items [a, b, c, d, exposure_rate]:")
print(np.round(item_bank.items[:3], 3))

In [ ]:
item = item_bank.items[0]
plot.item_curve(item[0], item[1], item[2], item[3], title="One Item")
plt.show()
plt.close("all")

## Run a small simulation

A simulation combines an initializer, selector, estimator, and stopper into a
`SimulationRunner`. The runner creates many CAT sessions and returns a
`SimulationResult`.

In [ ]:
runner = SimulationRunner(
  item_bank=item_bank,
  initializer=RandomInitializer(),
  selector=MaxInfoSelector(),
  estimator=NumericalSearchEstimator(),
  stopper=MinErrorStopper(0.35, min_items=6, max_items=18),
  seed=19,
)

result = runner.run(30)

print(f"Sessions: {len(result.sessions)}")
print(f"Bias: {result.bias:.3f}")
print(f"RMSE: {result.rmse:.3f}")
print(f"Mean test length: {np.mean([s.administered_count for s in result.sessions]):.2f}")

## Inspect one simulated session

`SimulationResult.sessions` gives access to the individual CAT traces. Each
session is a `CatSessionState`.

In [ ]:
session = result.sessions[0]
print("Session id:", session.session_id)
print("True theta:", round(float(session.true_theta), 3))
print("Final theta:", round(session.latest_theta, 3))
print("Items administered:", [int(item) for item in session.administered_item_ids[:8]], "...")
print("Responses:", session.responses[:8], "...")
print("Theta history:", [round(float(theta), 3) for theta in session.theta_history[:6]], "...")

## Plot one session and the item exposure profile

In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(9, 8))
plot.test_progress(simulation=result, index=0, info=True, see=True, ax=axes[0])
plot.item_exposure(simulation=result, par="b", ax=axes[1])
plt.tight_layout()
plt.show()
plt.close("all")

## Run one CAT session manually

`CatEngine` is the right entry point when another application controls the
session and provides real responses.

In [ ]:
engine = CatEngine(
  initializer=FixedPointInitializer(0.0),
  selector=MaxInfoSelector(),
  estimator=NumericalSearchEstimator(),
  stopper=TestLengthStopper(max_items=4),
)

context = RunContext(rng=np.random.default_rng(123))
manual_session = engine.start_session(session_id=1001, item_bank=item_bank, context=context)

mocked_responses = [True, False, True, True]
for response in mocked_responses:
  next_item = engine.select_next(manual_session, item_bank, context)
  step = engine.apply_response(manual_session, item_bank, next_item, response, context)
  print(
    f"item={int(step.selected_item_id):>3} response={step.response} "
    f"theta={float(step.updated_theta): .3f} stopped={step.stopped}"
  )
  if step.stopped:
    break

print("Final manual theta:", round(manual_session.latest_theta, 3))
print("Stop reason:", manual_session.stop_reason)

## Where to go next

- `02_manual_cat_sessions.ipynb` focuses on external-response workflows.
- `03_simulation_studies.ipynb` shows research-oriented experiments.
- `04_item_selection_and_stopping.ipynb` compares CAT strategies.
- `05_visualization_and_analysis.ipynb` focuses on figures and interpretation.